In [15]:
import uuid
from typing import TypedDict, Optional
from langgraph.graph import StateGraph
from langgraph.constants import START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

In [16]:
class State(TypedDict):
    username: Optional[str]
    email: Optional[str]

In [17]:
def registration_form(state: State) -> State:
    if not state.get("username"):
        username = interrupt("Please enter your username:")
    else:
        username = state["username"]

    if not state.get("email"):
        email = interrupt("Please enter your email:")
    else:
        email = state["email"]

    print(f"Username: {username}")
    print(f"Email: {email}")
    return {"username": username, "email": email}

In [18]:
builder = StateGraph(State)

builder.add_node("registration_form", registration_form)
builder.add_edge(START, "registration_form")
builder.add_edge("registration_form", END)

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [19]:
config = {
    "configurable": {
        "thread_id": uuid.uuid4()
    }
}

In [22]:
for step in graph.stream({"username": None, "email": None}, config):
    print(step)

{'__interrupt__': (Interrupt(value='Please enter your username:', resumable=True, ns=['registration_form:dbc94967-66b9-d708-5461-160a19ac9778']),)}


In [23]:
for step in graph.stream(Command(resume="pankaj"), config):
    print(step)

{'__interrupt__': (Interrupt(value='Please enter your email:', resumable=True, ns=['registration_form:dbc94967-66b9-d708-5461-160a19ac9778']),)}


In [24]:
for step in graph.stream(Command(resume="chandravanshi.pankaj@gmail.com"), config):
    print(step)

Username: pankaj
Email: chandravanshi.pankaj@gmail.com
{'registration_form': {'username': 'pankaj', 'email': 'chandravanshi.pankaj@gmail.com'}}


In [13]:
# First call: both fields are missing
for chunk in graph.stream({"username": "pankaj", "email": "pankaj@gmail.com"}, config):
    print(chunk)

Collected Username: pankaj
Collected Email: pankaj@gmail.com
{'collect_user_info': {'username': 'pankaj', 'email': 'pankaj@gmail.com'}}


In [26]:

for step in graph.stream({"username": None, "email": None}, config):
    print(step)
for step in graph.stream(Command(resume="pankaj", update={"username": "already_provided"}), config):
    print(step)

{'__interrupt__': (Interrupt(value='Please enter your username:', resumable=True, ns=['registration_form:a47f9a93-7d8e-d47e-f557-059ca9a5dc02']),)}
Username: already_provided
Email: pankaj
{'registration_form': {'username': 'already_provided', 'email': 'pankaj'}}
